In [0]:
from pyspark.sql import SparkSession
import pandas as pd

# Spark Session
spark = SparkSession.builder \
    .appName("Hospital Readmission Analysis") \
    .getOrCreate()

# Pandas తో load చేయండి
pandas_df = pd.read_csv("/Workspace/Shared/databricks_2027/cleaned_data.csv")

# PySpark DataFrame కి convert చేయండి
df = spark.createDataFrame(pandas_df)

print("✅ Data Loaded Successfully!")
print("Total Rows    :", df.count())
print("Total Columns :", len(df.columns))
df.show(5)

✅ Data Loaded Successfully!
Total Rows    : 101766
Total Columns : 46
+------------+-----------+---------------+------+-------+-----------------+------------------------+-------------------+----------------+------------------+--------------+---------------+-----------------+----------------+----------------+------+-------+-------+----------------+---------+-----------+-----------+--------------+-----------+-------------+---------+---------+-----------+------------+-------------+--------+--------+------------+----------+-------+-----------+-------+-------------------+-------------------+------------------------+-----------------------+----------------------+------+-----------+----------+-----------------+
|encounter_id|patient_nbr|           race|gender|    age|admission_type_id|discharge_disposition_id|admission_source_id|time_in_hospital|num_lab_procedures|num_procedures|num_medications|number_outpatient|number_emergency|number_inpatient|diag_1| diag_2| diag_3|number_diagnoses|metform

### Temp View Create

In [0]:
df.createOrReplaceTempView("patients")
print("✅ SQL Table Ready!")

✅ SQL Table Ready!


### Readmission Rate by Age Group

In [0]:
spark.sql("""
    SELECT age,
           COUNT(*) as total_patients,
           SUM(readmitted_binary) as readmitted_count,
           ROUND(SUM(readmitted_binary) * 100.0 / COUNT(*), 2) as readmission_rate
    FROM patients
    GROUP BY age
    ORDER BY readmission_rate DESC
""").show()

+--------+--------------+----------------+----------------+
|     age|total_patients|readmitted_count|readmission_rate|
+--------+--------------+----------------+----------------+
| [20-30)|          1657|             236|           14.24|
| [80-90)|         17197|            2078|           12.08|
| [70-80)|         26068|            3069|           11.77|
| [30-40)|          3775|             424|           11.23|
| [60-70)|         22483|            2502|           11.13|
|[90-100)|          2793|             310|           11.10|
| [40-50)|          9685|            1027|           10.60|
| [50-60)|         17256|            1668|            9.67|
| [10-20)|           691|              40|            5.79|
|  [0-10)|           161|               3|            1.86|
+--------+--------------+----------------+----------------+



###  Readmission Rate by Gender

In [0]:
spark.sql("""
    SELECT gender,
           COUNT(*) as total_patients,
           SUM(readmitted_binary) as readmitted_count,
           ROUND(SUM(readmitted_binary) * 100.0 / COUNT(*), 2) as readmission_rate
    FROM patients
    GROUP BY gender
    ORDER BY readmission_rate DESC
""").show()

+---------------+--------------+----------------+----------------+
|         gender|total_patients|readmitted_count|readmission_rate|
+---------------+--------------+----------------+----------------+
|         Female|         54708|            6152|           11.25|
|           Male|         47055|            5205|           11.06|
|Unknown/Invalid|             3|               0|            0.00|
+---------------+--------------+----------------+----------------+



### Avg Hospital Stay vs Readmission

In [0]:
spark.sql("""
    SELECT readmitted,
           COUNT(*) as total_patients,
           ROUND(AVG(time_in_hospital), 2) as avg_days,
           ROUND(AVG(num_medications), 2) as avg_medications,
           ROUND(AVG(number_diagnoses), 2) as avg_diagnoses
    FROM patients
    GROUP BY readmitted
    ORDER BY avg_days DESC
""").show()

+----------+--------------+--------+---------------+-------------+
|readmitted|total_patients|avg_days|avg_medications|avg_diagnoses|
+----------+--------------+--------+---------------+-------------+
|       <30|         11357|    4.77|           16.9|         7.69|
|       >30|         35545|     4.5|          16.28|         7.65|
|        NO|         54864|    4.25|          15.67|         7.22|
+----------+--------------+--------+---------------+-------------+



###  Top 10 High Risk Diagnoses

In [0]:
spark.sql("""
    SELECT diag_1,
           COUNT(*) as total_patients,
           SUM(readmitted_binary) as readmitted_count,
           ROUND(SUM(readmitted_binary) * 100.0 / COUNT(*), 2) as readmission_rate
    FROM patients
    WHERE diag_1 != 'Unknown'
    GROUP BY diag_1
    HAVING COUNT(*) > 100
    ORDER BY readmission_rate DESC
    LIMIT 10
""").show()

+------+--------------+----------------+----------------+
|diag_1|total_patients|readmitted_count|readmission_rate|
+------+--------------+----------------+----------------+
|   V58|           228|              95|           41.67|
|   443|           110|              24|           21.82|
|   593|           101|              21|           20.79|
|   572|           239|              49|           20.50|
|   202|           104|              20|           19.23|
| 250.7|           871|             165|           18.94|
|   790|           144|              27|           18.75|
| 250.6|          1183|             219|           18.51|
|   787|           267|              49|           18.35|
|   707|           257|              43|           16.73|
+------+--------------+----------------+----------------+



### High Risk Patients Identify


In [0]:
spark.sql("""
    SELECT age, gender, 
           time_in_hospital,
           num_medications,
           number_diagnoses,
           number_inpatient,
           readmitted
    FROM patients
    WHERE readmitted_binary = 1
    AND number_diagnoses >= 7
    AND num_medications >= 15
    ORDER BY number_diagnoses DESC, 
             num_medications DESC
    LIMIT 10
""").show()

+-------+------+----------------+---------------+----------------+----------------+----------+
|    age|gender|time_in_hospital|num_medications|number_diagnoses|number_inpatient|readmitted|
+-------+------+----------------+---------------+----------------+----------------+----------+
|[60-70)|  Male|               2|             28|              16|               0|       <30|
|[70-80)|Female|               4|             23|              16|               0|       <30|
|[80-90)|  Male|               5|             19|              16|               0|       <30|
|[60-70)|  Male|              11|             19|              16|               0|       <30|
|[60-70)|  Male|               7|             31|              15|               0|       <30|
|[70-80)|Female|               3|             17|              15|               1|       <30|
|[80-90)|  Male|               6|             27|              14|               2|       <30|
|[60-70)|  Male|               7|             29| 